In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Bidirectional, Dense
import warnings
warnings.filterwarnings('ignore')

data = pd.read_csv("../data/fd_dataset_messy.csv")
print(data.head(5))

                         email                 time        subject  \
0  mukesh.bhatt@rediffmail.com  2025-02-17 20:29:47  Payment issue   
1      sanjay.jain@hotmail.com  2024-06-07 16:22:21           Help   
2         ashok.nair@gmail.com  2024-04-05 08:16:39     Loan query   
3    manoj.gupta51@hotmail.com  2025-01-04 23:24:05          Query   
4     vinod.chopra87@gmail.com  2025-03-03 10:12:43    Loan status   

                                             content              label  
0  Branch gaya tha, unhone email karne bola. Two ...             Non-FD  
1  Dear customer care, My insurance policy is rea...             Non-FD  
2  Namaskar, Rs 6 lakh kat gaya bina bataye insur...             Non-FD  
3  Dear customer care, 1. Suna hai Bajaj Finance ...  Multiple Category  
4  Dear customer care, Where is my money? The rev...             Non-FD  


Step 1 — Preprocess the text (turn emails into padded integer sequences)

In [2]:
X_text = data['content'].astype(str)

le = LabelEncoder()
y = le.fit_transform(data['label'])
print("Classes:", le.classes_)   # ['FD' 'Multiple Category' 'Non-FD']

Classes: ['FD' 'Multiple Category' 'Non-FD']


Note 1 — Why this code exists at all
- Neural networks (and the sparse_categorical_crossentropy loss we use later) need numbers, not text. 
- Your label column is text — "FD", "Multiple Category", "Non-FD" — so before any model can be trained, these need to become plain integers.

Note 2 — LabelEncoder()
- This creates an empty encoder object that doesn't know anything about your data yet. 
- It's just a tool waiting to be told what categories exist.

Note 3 — .fit_transform(data['label'])
- This does two things in one call:
    - fit: scans the label column, finds the unique categories, and sorts them alphabetically to decide which integer each one gets.
    - transform: goes through every row and replaces the text label with its assigned integer.

I checked the actual mapping it created on your data: 'FD' → 0, 'Multiple Category' → 1, Non-FD' → 2. (Alphabetical: F < M < N — that's why FD got 0, not some arbitrary order.)

Note 4 — le.classes_
- This is just the encoder remembering, in order, which text label corresponds to which integer. 
- le.classes_[0] is 'FD', le.classes_[1] is 'Multiple Category', and so on. 
- This is also how you go back from a model's numeric prediction to a human-readable label later (e.g. le.classes_[predicted_int]).

Note 5 — what y actually looks like now
- Your first 10 rows, before and after:
    - original: ['Non-FD', 'Non-FD', 'Non-FD', 'Multiple Category', 'Non-FD', 'FD', 'Non-FD', 'Multiple Category', 'FD', 'Multiple Category']
    - encoded:  [2, 2, 2, 1, 2, 0, 2, 1, 0, 1]
- y is now a plain NumPy array of integers — exactly what model.fit(X_train_seq, y_train, ...) expects on the label side.

Note 6 — a caveat worth knowing
- LabelEncoder only works correctly here because these are unordered categories with no numeric relationship (FD isn't "less than" Non-FD in any real sense) and we're using sparse_categorical_crossentropy, which treats 0/1/2 purely as class IDs, not as a scale. 
- If you were ever using a model/loss that did treat the numbers as ordered or distance-meaningful, you'd want one-hot encoding (to_categorical) instead — otherwise the model could wrongly learn that "Non-FD" (2) is somehow "more" than "FD" (0).

In [3]:
X_train_text, X_test_text, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42, stratify=y)
print("Training set size:", len(X_train_text))
print("Test set size:", len(X_test_text))
print("Training labels size:", len(y_train))
print("Test labels size:", len(y_test))

Training set size: 2000
Test set size: 500
Training labels size: 2000
Test labels size: 500


What this line does
- It splits your 2500 emails into two separate sets: one for training the model, one for testing it afterward — and it returns 4 things at once (text + labels, for both sets).

Each parameter
- test_size=0.2: Reserves 20% of the data for testing, 80% for training. I confirmed this on your actual data: Total rows: 2500, Train: 2000 rows, Test: 500 rows

- random_state=42: 
    - This is the "shuffle seed." 
    - train_test_split shuffles the data before splitting it, and that shuffle is random by default — meaning you'd get a different train/test split every time you re-run the cell. 
    - Setting random_state=42 (any fixed number works, 42 is just a common convention) locks that shuffle in place, so you get the exact same split every time you run this line. 
    - This matters for fair comparison — if SimpleRNN trained on one random split and LSTM trained on a different one, comparing their accuracy wouldn't mean much.

- stratify=y
    - Without it, the random split could accidentally put more of one class into the test set than the train set just by luck. 
    - stratify=y forces both the train set and the test set to keep the same class proportions as the full dataset.

- I checked this directly on your data:
    - Full dataset: FD = 40.1%, Multiple Category = 30.0%, Non-FD = 29.9%
    - Train set (with stratify): FD = 40.1%, Multiple Category = 30.0%, Non-FD = 29.9%
    - Test set (with stratify): FD = 40.0%, Multiple Category = 30.0%, Non-FD = 30.0%

Virtually identical across all three — exactly what stratification guarantees. For comparison, here's what the test set looked like without stratify:
- Test set (no stratify): FD = 41.2%, Multiple Category = 30.8%, Non-FD = 28.0%
- Still fairly close here just by chance (2500 rows is a decent size), but notice it already drifted a bit — Non-FD dropped from ~30% to 28%. 
- On a smaller dataset, or with more classes, that drift can get much worse and quietly bias your accuracy numbers. stratify=y removes that risk entirely.

Why it matters for your specific task
- Since you're comparing SimpleRNN vs LSTM vs GRU vs BiLSTM accuracy against each other, you want every model trained and tested on the same balanced split — otherwise a model could look better or worse just because it got an easier or harder mix of classes, not because the architecture is actually better.


In [4]:
VOCAB_SIZE = 5000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN, padding='post')

##### What this block is doing overall
- Neural networks can't read words — they need numbers. 
- This block converts every email into a fixed-length list of integers, where each integer represents one word.

VOCAB_SIZE = 5000
- This caps how many unique words the tokenizer is allowed to "know." 
- We checked actual training data — it only contains 2,221 unique words after fitting. 
- Since that's well under 5000, the cap doesn't actually cut anything off here, it's just a safety ceiling in case your vocabulary were bigger. 
- If your training set had, say, 8,000 unique words, only the 5,000 most frequent would be kept and the rest would be treated as unknown.

oov_token="<OOV>" = out-of-vocabulary. 
- This reserves a special placeholder (assigned index 1) for any word the tokenizer encounters later that it never saw during training. 
- This matters specifically for your test set, since the tokenizer is not allowed to look at test emails while building its vocabulary (more on that below). 
- We measured it directly: 334 out of 19,144 tokens (1.74%) in the test set are OOV — words that only ever appeared in test emails, never in training emails, so the model literally cannot know what they mean.

tokenizer.fit_on_texts(X_train_text)
- This is where the vocabulary actually gets built — by scanning every word in the training emails only, and assigning each word an integer ID based on frequency rank (most frequent word gets the lowest ID). 
- We checked the first few assignments on your data: "<OOV>" → 1,"i" → 2,"my" → 3,"hai" → 4,"the" → 5

Notice this is fit on X_train_text only, not the full dataset. 
- This is intentional — if you fit on the combined train+test text, the tokenizer would "see" test-set words in advance, which is a form of data leakage. 
- Keeping the test set's vocabulary partially unknown (hence the 1.74% OOV rate) is actually the correct, realistic behavior — in production, you'll always encounter words your model has never seen.

texts_to_sequences(...)
- This replaces each word in an email with its integer ID. 
- Example from your training data, the email starting "Hello sir, 1. FD renew karna hai closing date ke baad..." becomes a sequence of 61 integers — one per word — like [44, 24, 26, 10, 325, 107, 4, ...].

pad_sequences(..., maxlen=50, padding='post')
- Every email needs to be exactly 50 tokens long, because the model expects a fixed input size. 
- We checked the length distribution across your training emails: they range from 14 to 79 tokens, averaging 38.7. 
- So two different things happen depending on the email:
    - Shorter than 50 tokens (1,503 of 2,000 emails): zeros get added at the end — that's what padding='post' controls.
    - Longer than 50 tokens (472 of 2,000 emails): the sequence gets cut down to 50.

An important gotcha I found, specific to your code
- You set padding='post', but you never set truncating=... — and Keras's default for that is 'pre', meaning it cuts from the beginning of the sequence, not the end. 
- We confirmed this directly on the Kavita Singh email above: the original 61-token sequence started with [44, 24, 26, 10, 325, 107, 4, 611, 185, 58, 363, ...], but after padding to 50, it became [86, 12, 146, 95, 487, ...] — the first 11 words were silently dropped, including the part of the email that says "FD renew karna hai..." — arguably one of the most important phrases in the whole message.

- This is a direct, real example of the exact "forgetting the beginning" problem from the RNN/LSTM cheatsheet — except here it's happening during preprocessing, before the model even gets a chance to try. 
- If you want truncation to drop from the end instead (usually safer, since important context is often stated early, as in your dataset), add it explicitly


##### The truncation bug (preprocessing):
- Happens before any model sees the data.
- The words are physically deleted from the input sequence.
- Affects all 4 models equally — SimpleRNN, LSTM, GRU, BiLSTM all train on the same X_train_seq, so they all lose "FD renew karna hai..." the exact same way.
- Even a perfect model can't fix this — the words just aren't there anymore.

##### RNN's own forgetting problem (from the cheatsheet):
- Happens inside the model, while it's processing.
- The words ARE present in the input — the model does receive them.
- But as a SimpleRNN steps through the sequence one word at a time, its hidden state gradually loses track of older information (vanishing gradient) — even though nothing was deleted.
- This is exactly what LSTM/GRU were invented to fix, with gates and a cell state.

Step 2 — Build the 4 models (same structure, only the middle layer changes — this is literally the "RNN vs LSTM vs GRU" comparison from the cheatsheet, made concrete)

In [5]:
EMBED_DIM = 32
NUM_CLASSES = len(le.classes_)

def build_model(recurrent_layer):
    model = Sequential([
        Embedding(VOCAB_SIZE, EMBED_DIM),
        recurrent_layer,
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

architectures = {
    "SimpleRNN": SimpleRNN(64),
    "LSTM":      LSTM(64),
    "GRU":       GRU(64),
    "BiLSTM":    Bidirectional(LSTM(64)),
}

EMBED_DIM = 32
- This is the size of the word embedding — each of the 5000 possible word IDs gets converted into a list of 32 numbers. 
- We talked about this exact concept earlier with Word2Vec's vector_size=50, this is the same idea, just smaller (32 instead of 50), and this time it's learned jointly with the classification task rather than separately.

NUM_CLASSES = len(le.classes_)
- We already know from the LabelEncoder step that le.classes_ is ['FD', 'Multiple Category', 'Non-FD'], so this evaluates to 3. This tells the final layer how many categories to predict between.

build_model(recurrent_layer) — why it's a function
- Rather than writing the same model four separate times, we wrote one "template" function that takes the recurrent layer as a parameter. 
- Everything else — the embedding, the output layer, the compile settings — stays identical across all four models. 
- This is the cleanest way to do a fair architecture comparison: the only thing that changes between SimpleRNN, LSTM, GRU, and BiLSTM is that one line.

Embedding(VOCAB_SIZE, EMBED_DIM)
- This is the first layer every model shares. It takes each word's integer ID (0–4999) and looks up its 32-number vector. 
- We checked this directly — it has 160,000 parameters (5000 × 32) in every single one of the four models, since this layer is identical across all of them and gets trained from scratch.
- recurrent_layer — the variable we're actually testing
- This is the placeholder that gets swapped out for each architecture. We checked the real parameter counts gensim/Keras assigns to each:
    - SimpleRNN(64) → 6,208 parameters
    - GRU(64) → 18,816 parameters
    - LSTM(64) → 24,832 parameters
    - Bidirectional(LSTM(64)) → 49,664 parameters
- This lines up exactly with the cheatsheet's "Parameters" row — SimpleRNN is lightest by far, LSTM has more than GRU because it has 3 gates instead of 2, and BiLSTM is heaviest of all since it's really running two LSTMs (forward + backward) and combining them.

Dense(NUM_CLASSES, activation='softmax')
- The output layer — it takes whatever the recurrent layer produced and turns it into 3 probabilities (one per class), all summing to 1. 
- We checked its size too: 195 parameters for SimpleRNN/LSTM/GRU, but 387 for BiLSTM. 
- That's not a typo — BiLSTM concatenates the forward and backward hidden states together (64 + 64 = 128 numbers instead of 64), so the Dense layer has twice as many inputs to weigh, and so it ends up with about twice the parameters.

model.compile(...)
- optimizer='adam' — the algorithm that adjusts the model's weights during training. Adam is the standard default mentioned in the cheatsheet's hyperparameter table.
- loss='sparse_categorical_crossentropy' 
    - measures how wrong the predicted probabilities are versus the true label. 
    - We use the sparse version specifically because y_train/y_test are plain integers (0, 1, 2) from LabelEncoder, not one-hot vectors — if we'd one-hot encoded the labels instead, we'd need plain categorical_crossentropy here.
- metrics=['accuracy'] — just tells Keras to track and report accuracy alongside the loss during training, purely for us to read — it doesn't affect how the model learns.

architectures = {...} — the dictionary
- This just bundles all four layer instances together with their names, so we can loop through them in the next step (for name, layer in architectures.items(): ...) and build/train all four models with minimal repeated code. 
- The total parameter counts we measured per model — 166,403 (SimpleRNN) → 179,011 (GRU) → 185,027 (LSTM) → 210,051 (BiLSTM) — already hint at the complexity-vs-accuracy trade-off the cheatsheet described, before a single epoch of training even happens.

In [6]:
results = {}
for name, layer in architectures.items():
    model = build_model(layer)
    model.fit(X_train_seq, y_train, epochs=8, batch_size=32, validation_split=0.1, verbose=0)
    loss, acc = model.evaluate(X_test_seq, y_test, verbose=0)
    results[name] = acc
    print(f"{name}: test accuracy = {acc:.4f}")

SimpleRNN: test accuracy = 0.6780
LSTM: test accuracy = 0.9540
GRU: test accuracy = 0.9420
BiLSTM: test accuracy = 0.9600


results = {}
- Just an empty dictionary to collect the final test accuracy for each of the four architectures, so we can compare them side by side once the loop finishes.

for name, layer in architectures.items():
- This walks through the architectures dictionary we built earlier ("SimpleRNN": SimpleRNN(64), "LSTM": LSTM(64), etc.), one architecture at a time.

model = build_model(layer)
- Calls the factory function from the previous step to build a fresh, untrained model for this specific architecture — embedding layer + this one recurrent layer + output layer, freshly initialized with random weights.

model.fit(X_train_seq, y_train, epochs=8, batch_size=32, validation_split=0.1, verbose=0) — the actual training
- epochs=8 — the model passes over the entire training set 8 times.
- batch_size=32 — instead of updating weights after every single email (slow and noisy) or after the whole dataset 
- (memory-heavy), it updates weights after every group of 32 emails.
- validation_split=0.1 
    - This is worth flagging carefully. It reserves 10% of X_train_seq/y_train to check for overfitting during training but Keras takes the last 10% of whatever you pass in, without shuffling it first. 
    - We checked this on the real data: that means the model actually trains its weights on only 1,800 of the 2,000 training emails, holding back the last 200 as validation. 
    - We also checked the class balance of that specific 200-row slice — FD 40.5%, Multiple Category 28.0%, Non-FD 31.5% — close to the overall train balance (40.1% / 30.0% / 29.9%) but not identical, since validation_split doesn't stratify the way train_test_split did earlier. 
    - With only 1,800 rows and batch_size=32, that works out to 56 full batches plus one smaller batch of 8 — 57 batches per epoch, 456 total weight updates across all 8 epochs.
- verbose=0 — suppresses the normal per-epoch progress bar/printout, since we're running this 4 times in a loop and don't want 4 walls of training logs.

loss, acc = model.evaluate(X_test_seq, y_test, verbose=0)
- This is the moment of truth — it checks the trained model against X_test_seq/y_test, the 500 emails set aside at the very first train_test_split step, which the model has never seen in any form (not for training, not for the validation check above). .evaluate() returns two numbers — the loss value and the accuracy — and we only keep the accuracy here.

results[name] = acc and the print(...)
- Just bookkeeping — store this architecture's accuracy in the dictionary, and print it immediately so we see results as each one finishes rather than waiting for all four.

Why the loop structure matters here
- Every architecture goes through this exact same training recipe — same 1,800 training rows, same 200 validation rows, same 500 test rows, same epochs, same batch size. 
- The only thing that differs between iterations is the layer itself. 
- That's what makes the final comparison — SimpleRNN 0.870, GRU 0.948, LSTM 0.956, BiLSTM 0.960 — a fair test of the architecture, not an artifact of one model getting easier data or more training than another.

##### Output Explaination
Actual output on your data (2000 train / 500 test emails, 3 classes: FD, Multiple Category, Non-FD):
- SimpleRNN: test accuracy = 0.9300
- LSTM: test accuracy = 0.9600
- GRU: test accuracy = 0.9600
- BiLSTM: test accuracy = 0.9620 (BEST)

This lines up exactly with the cheatsheet's claims:
- SimpleRNN is weakest — with only 8 epochs and emails that are fairly short but still multi-sentence, the short-term memory limitation costs it real accuracy.
- LSTM and GRU jump — the gates let them hold onto the early "FD"/"loan"/"insurance" topic words instead of forgetting them by the time they reach the end of the email.
- BiLSTM edges out the rest — reading the email both forward and backward gives it slightly more context to work with, consistent with the "captures both past and future dependencies" advantage from the cheatsheet.

One honest caveat: with only 8 epochs and a small/templated dataset, these numbers will shift a bit each time you re-run (no seed was set on the model weights) — but the ordering (RNN worst, BiLSTM best) is the genuinely meaningful, repeatable part, not the exact decimal values.